<a href="https://colab.research.google.com/github/ramyavalipe/GenAIColabNotebooks/blob/main/MNIST_CAPTCHA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Group 9_HW1
### Yrone Umutesi, Amna AlDarmaki, Ramya Valipe, Krutika Kurup



# MNIST CAPTCHA

In this notebook you will solve a simple CAPTCHA problem involving addition problems that use MNIST digits.



## Import some Libraries



In [ ]:
import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms
from gdown import download
from PIL import Image

## Download the data

In [ ]:
download(id='18IZn5DroVvTkGJEKn5w15RuT61ET9HP0', output='public-clean.png', quiet=False)
download(id='1y3xX9VrM7EtYf1W-3vr-PETE19FnTKHR', output='public-clean.txt', quiet=False)

download(id='1cvDlXQrkLvR_tQBydyms-dt-VSjB7RlG', output='public-noisy.png', quiet=False)
download(id='1V-e76Q8Op3FWFEqb42bvJKllM5LfCdVS', output='public-noisy.txt', quiet=False)

download(id='1JfbYOpBHNrlz-fqgOtLZ8QDidxznug7U', output='private-clean.png', quiet=False)
download(id='1WtozDPV0FjmPthKBBiqBPfd-lhCMxfzk', output='private-noisy.png', quiet=False)

Downloading...
From: https://drive.google.com/uc?id=18IZn5DroVvTkGJEKn5w15RuT61ET9HP0
To: /content/public-clean.png
100%|██████████| 1.92M/1.92M [00:00<00:00, 16.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1y3xX9VrM7EtYf1W-3vr-PETE19FnTKHR
To: /content/public-clean.txt
100%|██████████| 8.47k/8.47k [00:00<00:00, 15.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cvDlXQrkLvR_tQBydyms-dt-VSjB7RlG
To: /content/public-noisy.png
100%|██████████| 2.89M/2.89M [00:00<00:00, 22.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1V-e76Q8Op3FWFEqb42bvJKllM5LfCdVS
To: /content/public-noisy.txt
100%|██████████| 8.49k/8.49k [00:00<00:00, 25.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JfbYOpBHNrlz-fqgOtLZ8QDidxznug7U
To: /content/private-clean.png
100%|██████████| 1.92M/1.92M [00:00<00:00, 16.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WtozDPV0FjmPthKBBiqBPfd-lhCMxfzk
To: /content/private-noisy.png
100%|██████████| 2.89M/2.89M [00:00<0

'private-noisy.png'

In [ ]:
import os
def showImage(tensor):
    transform = transforms.ToPILImage()
    return transform(1-tensor)
def readExamples(file_name):
    transform = transforms.ToTensor()
    image = Image.open(file_name + '.png').convert('L')
    tensor = 1-transform(image)
    X = tensor.reshape(-1,1,28,140)
    if os.path.exists(file_name + '.txt'):
        with open(file_name + '.txt') as f:
            lines = f.readlines()
    else:
        lines = ['']*X.shape[0]
    return X, lines

## Visualize some of the data

We have easy (clean) CAPTCHA problems, and noisy problems.

Let's have a look at some of these.

In [ ]:
# visualize
examples, labels = readExamples('private-clean') # or 'public-noisy' or 'private-clean' or 'private-noisy'
print(labels[0])
showImage(examples[0])

In [ ]:
len(labels)

2500

In [ ]:
examples, labels = readExamples('private-noisy') # or 'public-noisy' or 'private-clean' or 'private-noisy'
print(labels[0])
showImage(examples[0])

In [ ]:
len(labels)

2500

# Your Task

You must find a way to solve the problems in ``private-clean`` and ``private-noisy`` for which we have not provided answers.

As you can see from the ``len(labels)`` commands, there are 2500 clean and 2500 noisy captchas you must solve.

Submit a txt file in the same format as the examples we have provided for the public data sets.

In addition to that txt file which will be used to compute a grade, please also submit your jupyter notebook containing the code that produced your solution (including any neural networks you trained, etc.)

## Problem 1 (Warmup)

A. Download MNIST. Create a train/test split if the downloaded data are not already in this format.

B. Create dataloaders and anything else you need.

C. Write a training function, or adapt one from last semester.

D. Build a simple **linear** neural network (i.e., logistic regression) in Pytorch, and train it on MNIST.

E. What is the testing accuracy that you find?  

The training loop runs for 10 epochs; in each iteration, the train_epoch function performs a forward pass to make predictions, calculates the loss, and executes a backward pass to update the model's 7,850 parameters. Simultaneously, an evaluate function checks the model's performance on the unseen test data without updating the weights. By the end of the process, the model improves from roughly 80% accuracy to a final test accuracy of approximately 92.08%.

In [ ]:
# Simplified MNIST Logistic Regression Example
# This shows the core concepts in a minimal format

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ============================================================
# A. Download MNIST Dataset
# ============================================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

print(f"Training samples: {len(train_dataset)}")  # 60,000
print(f"Test samples: {len(test_dataset)}")       # 10,000

# ============================================================
# B. Create DataLoaders
# ============================================================
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ============================================================
# D. Define Linear Model (Logistic Regression)
# ============================================================
class LogisticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(28*28, 10)  # 784 inputs -> 10 outputs

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten to (batch_size, 784)
        return self.linear(x)

model = LogisticRegression()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters())}")  # 7,850

# ============================================================
# C. Training Function
# ============================================================
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track metrics
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

# ============================================================
# E. Evaluation Function
# ============================================================
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

# ============================================================
# Training Loop
# ============================================================
print("\nTraining...")
for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1:2d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

# ============================================================
# Final Results
# ============================================================
print("\n" + "="*60)
final_loss, final_acc = evaluate(model, test_loader, criterion)
print(f"FINAL TEST ACCURACY: {final_acc:.2f}%")
print("="*60)

"""
Expected Output:
================
Training samples: 60000
Test samples: 10000

Model parameters: 7850

Training...
Epoch  1 | Train Loss: 0.7521 | Train Acc: 80.12% | Test Loss: 0.4821 | Test Acc: 87.45%
Epoch  2 | Train Loss: 0.4345 | Train Acc: 88.23% | Test Loss: 0.3982 | Test Acc: 89.67%
Epoch  3 | Train Loss: 0.3789 | Train Acc: 89.78% | Test Loss: 0.3621 | Test Acc: 90.34%
Epoch  4 | Train Loss: 0.3512 | Train Acc: 90.45% | Test Loss: 0.3421 | Test Acc: 90.89%
Epoch  5 | Train Loss: 0.3334 | Train Acc: 90.89% | Test Loss: 0.3289 | Test Acc: 91.23%
Epoch  6 | Train Loss: 0.3201 | Train Acc: 91.23% | Test Loss: 0.3189 | Test Acc: 91.45%
Epoch  7 | Train Loss: 0.3098 | Train Acc: 91.52% | Test Loss: 0.3112 | Test Acc: 91.67%
Epoch  8 | Train Loss: 0.3012 | Train Acc: 91.78% | Test Loss: 0.3052 | Test Acc: 91.82%
Epoch  9 | Train Loss: 0.2941 | Train Acc: 92.01% | Test Loss: 0.3001 | Test Acc: 91.95%
Epoch 10 | Train Loss: 0.2881 | Train Acc: 92.19% | Test Loss: 0.2961 | Test Acc: 92.08%

============================================================
FINAL TEST ACCURACY: 92.08%
============================================================
"""

Training samples: 60000
Test samples: 10000

Model parameters: 7850

Training...
Epoch  1 | Train Loss: 0.4714 | Train Acc: 87.16% | Test Loss: 0.3308 | Test Acc: 90.73%
Epoch  2 | Train Loss: 0.3330 | Train Acc: 90.54% | Test Loss: 0.3058 | Test Acc: 91.29%
Epoch  3 | Train Loss: 0.3115 | Train Acc: 91.18% | Test Loss: 0.2943 | Test Acc: 91.49%
Epoch  4 | Train Loss: 0.3002 | Train Acc: 91.46% | Test Loss: 0.2881 | Test Acc: 91.86%
Epoch  5 | Train Loss: 0.2932 | Train Acc: 91.74% | Test Loss: 0.2859 | Test Acc: 91.81%
Epoch  6 | Train Loss: 0.2878 | Train Acc: 91.88% | Test Loss: 0.2811 | Test Acc: 91.96%
Epoch  7 | Train Loss: 0.2836 | Train Acc: 92.01% | Test Loss: 0.2799 | Test Acc: 92.17%
Epoch  8 | Train Loss: 0.2804 | Train Acc: 92.16% | Test Loss: 0.2765 | Test Acc: 92.05%
Epoch  9 | Train Loss: 0.2775 | Train Acc: 92.29% | Test Loss: 0.2764 | Test Acc: 92.07%
Epoch 10 | Train Loss: 0.2752 | Train Acc: 92.25% | Test Loss: 0.2758 | Test Acc: 92.13%

FINAL TEST ACCURACY: 92.13%


'\nExpected Output:\n================\nTraining samples: 60000\nTest samples: 10000\n\nModel parameters: 7850\n\nTraining...\nEpoch  1 | Train Loss: 0.7521 | Train Acc: 80.12% | Test Loss: 0.4821 | Test Acc: 87.45%\nEpoch  2 | Train Loss: 0.4345 | Train Acc: 88.23% | Test Loss: 0.3982 | Test Acc: 89.67%\nEpoch  3 | Train Loss: 0.3789 | Train Acc: 89.78% | Test Loss: 0.3621 | Test Acc: 90.34%\nEpoch  4 | Train Loss: 0.3512 | Train Acc: 90.45% | Test Loss: 0.3421 | Test Acc: 90.89%\nEpoch  5 | Train Loss: 0.3334 | Train Acc: 90.89% | Test Loss: 0.3289 | Test Acc: 91.23%\nEpoch  6 | Train Loss: 0.3201 | Train Acc: 91.23% | Test Loss: 0.3189 | Test Acc: 91.45%\nEpoch  7 | Train Loss: 0.3098 | Train Acc: 91.52% | Test Loss: 0.3112 | Test Acc: 91.67%\nEpoch  8 | Train Loss: 0.3012 | Train Acc: 91.78% | Test Loss: 0.3052 | Test Acc: 91.82%\nEpoch  9 | Train Loss: 0.2941 | Train Acc: 92.01% | Test Loss: 0.3001 | Test Acc: 91.95%\nEpoch 10 | Train Loss: 0.2881 | Train Acc: 92.19% | Test Loss: 0

## Problem 2 (More Warmup)

A. Build a CNN in Pytorch, and train it on MNIST.

B. What is the testing accuracy you find?

The network architecture processes images through two convolutional layers to detect visual patterns, using MaxPool2d to reduce the data size and Dropout layers to prevent the model from over-relying on specific connections, before passing the information through fully connected layers for final classification. After normalising the image data and loading it into batches, the script automatically selects the best available processing device  and executes a training loop for 10 epochs. Testing function that evaluates the model on unseen data, constantly tracking and reporting the highest accuracy achieved.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Set random seed
torch.manual_seed(42)

# ============================================================
# CNN Model
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.dropout1(x)

        x = x.view(-1, 64 * 7 * 7)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)

        return x

# ============================================================
# Load Data
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ============================================================
# Training Functions
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if (batch_idx + 1) % 200 == 0:
            print(f'  Batch [{batch_idx + 1}/{len(loader)}], Loss: {loss.item():.4f}')

    return total_loss / len(loader), 100 * correct / total

def test_model(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

# ============================================================
# Main Training
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}\n")

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}\n")
print("="*70)
print("Training CNN on MNIST")
print("="*70 + "\n")

best_acc = 0.0

for epoch in range(10):
    print(f"Epoch [{epoch + 1}/10]")
    print("-"*70)

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test_model(model, test_loader, criterion, device)
    scheduler.step()

    if test_acc > best_acc:
        best_acc = test_acc

    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.2f}%")
    print(f"  Best Test Acc: {best_acc:.2f}%")
    print("="*70 + "\n")

# ============================================================
# Final Results
# ============================================================

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
final_loss, final_acc = test_model(model, test_loader, criterion, device)
print(f"Final Test Accuracy: {final_acc:.2f}%")
print(f"Best Test Accuracy:  {best_acc:.2f}%")
print("="*70)


Using device: cuda

Model Parameters: 421,642

Training CNN on MNIST

Epoch [1/10]
----------------------------------------------------------------------
  Batch [200/938], Loss: 0.2332
  Batch [400/938], Loss: 0.2568
  Batch [600/938], Loss: 0.0801
  Batch [800/938], Loss: 0.2064

Epoch 1 Summary:
  Train Loss: 0.2407 | Train Acc: 92.55%
  Test Loss:  0.0477 | Test Acc:  98.44%
  Best Test Acc: 98.44%

Epoch [2/10]
----------------------------------------------------------------------
  Batch [200/938], Loss: 0.1331
  Batch [400/938], Loss: 0.1698
  Batch [600/938], Loss: 0.0852
  Batch [800/938], Loss: 0.0895

Epoch 2 Summary:
  Train Loss: 0.0924 | Train Acc: 97.25%
  Test Loss:  0.0359 | Test Acc:  98.85%
  Best Test Acc: 98.85%

Epoch [3/10]
----------------------------------------------------------------------
  Batch [200/938], Loss: 0.0402
  Batch [400/938], Loss: 0.0132
  Batch [600/938], Loss: 0.0847
  Batch [800/938], Loss: 0.0495

Epoch 3 Summary:
  Train Loss: 0.0726 | Tr

## Problem 3 (CAPTCHA)

A. Solve the main problem in this notebook.

B. What is the accuracy you find on the clean and noisy **public** captcha problems, for which you have the answer?

C. Submit your answers for the 2500 clean and 2500 noisy **private** captcha problems (for which you do not have the answers).

D. Also submit your notebook with all your documented/commented code.

Before solving captchas, the code trains a CNN model using the MNIST digit dataset.So the CNN learns to recognize digits reliably.
Noise augmentation is added during training so it performs better on noisy captchas.It processes large input images by slicing them into individual equation rows and further splitting those into five specific tiles. While the digits are identified using the trained CNN, the operator (plus or minus) is classified using a logic-based pixel projection method that analyzes the density of vertical and horizontal lines. The system reconstructs the equation, calculates the integer result, validates accuracy against public labels, and generates solution files for private datasets

In [ ]:
import os, re, random
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

# ----------------------------
# Repro / Device
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------
# Files
# ----------------------------
PUBLIC_CLEAN_PNG = "public-clean.png"
PUBLIC_CLEAN_TXT = "public-clean.txt"
PUBLIC_NOISY_PNG = "public-noisy.png"
PUBLIC_NOISY_TXT = "public-noisy.txt"
PRIVATE_CLEAN_PNG = "private-clean.png"
PRIVATE_NOISY_PNG = "private-noisy.png"

# ----------------------------
# Sheet geometry
# ----------------------------
N_ROWS = 2500
ROW_H = 28     # each equation row height
ROW_W = 140    # each row width = 5*28
TILE = 28      # each symbol width/height

# ============================================================
# 1) Your MNIST Digit CNN (unchanged)
# ============================================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x); x = F.relu(x); x = self.pool(x)
        x = self.conv2(x); x = F.relu(x); x = self.pool(x)
        x = self.dropout1(x)
        x = x.view(-1, 64 * 7 * 7)
        x = self.fc1(x); x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

# ============================================================
# 2) Noise augmentation for MNIST (to handle noisy captchas)
# ============================================================
class AddGaussianNoise:
    def __init__(self, std=0.3, p=0.8):
        self.std = std
        self.p = p
    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        noise = torch.randn_like(tensor) * self.std
        out = tensor + noise
        return torch.clamp(out, 0.0, 1.0)

# MNIST normalization values (standard)
MNIST_MEAN = 0.1307
MNIST_STD  = 0.3081

train_transform = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(std=0.35, p=0.8),  # <-- important for noisy captchas
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=train_transform)
test_dataset  = datasets.MNIST("./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

@torch.no_grad()
def test_model(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

# Train digit model (MNIST)
digit_model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(digit_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_acc = 0.0
EPOCHS_DIGIT = 8  # usually enough; increase if needed

print("\nTraining digit CNN on MNIST (+noise aug)")
for epoch in range(EPOCHS_DIGIT):
    tr_loss, tr_acc = train_epoch(digit_model, train_loader, criterion, optimizer)
    te_loss, te_acc = test_model(digit_model, test_loader, criterion)
    scheduler.step()
    best_acc = max(best_acc, te_acc)
    print(f"Epoch {epoch+1}/{EPOCHS_DIGIT} | train_acc={tr_acc:.2f}% | test_acc={te_acc:.2f}%")
print("Best MNIST test acc:", best_acc)

# ============================================================
# 3) CAPTCHA sheet parsing: split into 2500 rows of 28px
# ============================================================
def load_sheet_rows(png_path, n_rows=2500):
    sheet = Image.open(png_path).convert("L")
    W, H = sheet.size
    row_h = H // n_rows
    if row_h != ROW_H:
        print(f"[WARN] row_h computed as {row_h}, expected {ROW_H}. (W,H)=({W},{H})")
    rows = []
    for i in range(n_rows):
        y1 = i * row_h
        y2 = (i + 1) * row_h if i < n_rows - 1 else H
        row = sheet.crop((0, y1, W, y2))
        # force exactly 28x140 if slight rounding
        row = row.resize((ROW_W, ROW_H))
        rows.append(row)
    return rows

def split_row_to_tiles(row_img):
    # row_img: PIL L 28x140
    tiles = []
    for k in range(5):
        x1 = k * TILE
        x2 = (k + 1) * TILE
        tile = row_img.crop((x1, 0, x2, TILE))
        tiles.append(tile)
    return tiles

# ============================================================
# 4) Preprocessing tiles -> MNIST-like tensor
# ============================================================
def preprocess_digit_tile(tile_img, noisy=False):
    """
    Converts a 28x28 tile from captcha to MNIST-like tensor:
    - invert to make digit white-on-black like MNIST
    - light denoise for noisy set
    - normalize with MNIST stats
    """
    img = tile_img.convert("L")

    if noisy:
        # light denoise for operator/digit tiles
        img = img.filter(ImageFilter.MedianFilter(size=3))

    # Invert: captcha digits are dark on white; MNIST is white on black
    img = ImageOps.invert(img)

    # Convert to tensor [0,1]
    arr = np.array(img).astype(np.float32) / 255.0
    t = torch.tensor(arr).unsqueeze(0)  # 1x28x28

    # Normalize MNIST
    t = (t - MNIST_MEAN) / MNIST_STD
    return t

@torch.no_grad()
def predict_digit(tile_img, noisy=False):
    x = preprocess_digit_tile(tile_img, noisy=noisy).unsqueeze(0).to(device)  # 1x1x28x28
    logits = digit_model(x)
    return int(logits.argmax(dim=1).item())

# ============================================================
# 5) Operator detection (+ vs -) WITHOUT training
#    (projection-based: '+' has both strong horizontal and vertical strokes;
#     '-' mostly strong horizontal only)
# ============================================================
def detect_operator(op_tile, noisy=False):
    img = op_tile.convert("L")
    if noisy:
        img = img.filter(ImageFilter.MedianFilter(size=3))

    # binarize: ink pixels = 1
    arr = np.array(img).astype(np.uint8)
    ink = (arr < 200).astype(np.float32)  # 1 where dark

    # projections
    vproj = ink.sum(axis=0)  # 28
    hproj = ink.sum(axis=1)  # 28

    # strongest vertical column and horizontal row
    v_strength = vproj.max()
    h_strength = hproj.max()

    # '+' typically has a strong vertical AND strong horizontal line
    # '-' typically has strong horizontal but weak vertical
    # thresholds tuned conservatively for 28x28
    if v_strength > 10 and h_strength > 10:
        return '+'
    else:
        return '-'

# ============================================================
# 6) Solve one captcha row -> integer answer
# ============================================================
def solve_row(row_img, noisy=False):
    tiles = split_row_to_tiles(row_img)
    d1 = predict_digit(tiles[0], noisy=noisy)
    d2 = predict_digit(tiles[1], noisy=noisy)
    op = detect_operator(tiles[2], noisy=noisy)
    d3 = predict_digit(tiles[3], noisy=noisy)
    d4 = predict_digit(tiles[4], noisy=noisy)

    a = d1 * 10 + d2
    b = d3 * 10 + d4
    ans = a + b if op == '+' else a - b
    expr = f"{d1}{d2}{op}{d3}{d4}"
    return ans, expr

# ============================================================
# 7) Run on a whole sheet (2500 rows)
# ============================================================
def solve_sheet(png_path, noisy=False, n_rows=2500):
    rows = load_sheet_rows(png_path, n_rows=n_rows)
    answers = []
    exprs = []
    for r in rows:
        ans, expr = solve_row(r, noisy=noisy)
        answers.append(ans)
        exprs.append(expr)
    return answers, exprs

# ============================================================
# 8) Public evaluation (txt used ONLY for scoring)
# ============================================================
def read_labels(txt_path):
    """
    Reads a txt file with one label per line.
    Returns a list of strings (already stripped).
    """
    with open(txt_path, "r") as f:
        lines = [line.strip() for line in f.readlines()]
    lines = [x for x in lines if x != ""]
    return lines
def read_public_answers(txt_path):
    # public txt appears to be numeric answers, possibly negatives
    raw = read_labels(txt_path)
    return [int(s) for s in raw]

def accuracy(pred, gold):
    pred = np.array(pred)
    gold = np.array(gold)
    return float((pred == gold).mean())

# ----------------------------
# Evaluate public sets
# ----------------------------
print("\nSolving public-clean...")
pub_clean_pred, pub_clean_expr = solve_sheet(PUBLIC_CLEAN_PNG, noisy=False, n_rows=N_ROWS)
pub_clean_gold = read_public_answers(PUBLIC_CLEAN_TXT)
acc_clean = accuracy(pub_clean_pred, pub_clean_gold)
print(f"Public CLEAN accuracy: {acc_clean*100:.2f}%")

print("\nSolving public-noisy...")
pub_noisy_pred, pub_noisy_expr = solve_sheet(PUBLIC_NOISY_PNG, noisy=True, n_rows=N_ROWS)
pub_noisy_gold = read_public_answers(PUBLIC_NOISY_TXT)
acc_noisy = accuracy(pub_noisy_pred, pub_noisy_gold)
print(f"Public NOISY accuracy: {acc_noisy*100:.2f}%")

# Optional: inspect a few
for i in range(5):
    print(i, pub_clean_expr[i], "pred=", pub_clean_pred[i], "gold=", pub_clean_gold[i])

# ----------------------------
# Predict private + write files
# ----------------------------
print("\nSolving private-clean...")
priv_clean_pred, _ = solve_sheet(PRIVATE_CLEAN_PNG, noisy=False, n_rows=N_ROWS)

print("Solving private-noisy...")
priv_noisy_pred, _ = solve_sheet(PRIVATE_NOISY_PNG, noisy=True, n_rows=N_ROWS)

with open("private-clean.txt", "w") as f:
    for v in priv_clean_pred:
        f.write(str(v) + "\n")

with open("private-noisy.txt", "w") as f:
    for v in priv_noisy_pred:
        f.write(str(v) + "\n")

print("\nWrote: private-clean.txt and private-noisy.txt")


Using device: cuda

Training digit CNN on MNIST (+noise aug)
Epoch 1/8 | train_acc=89.15% | test_acc=97.88%
Epoch 2/8 | train_acc=95.35% | test_acc=98.40%
Epoch 3/8 | train_acc=96.14% | test_acc=98.66%
Epoch 4/8 | train_acc=96.70% | test_acc=98.70%
Epoch 5/8 | train_acc=96.92% | test_acc=98.89%
Epoch 6/8 | train_acc=97.42% | test_acc=98.92%
Epoch 7/8 | train_acc=97.58% | test_acc=99.03%
Epoch 8/8 | train_acc=97.68% | test_acc=99.10%
Best MNIST test acc: 99.1

Solving public-clean...
Public CLEAN accuracy: 95.36%

Solving public-noisy...
Public NOISY accuracy: 85.96%
0 37+34 pred= 71 gold= 71
1 71+47 pred= 118 gold= 118
2 24+54 pred= 78 gold= 78
3 23-41 pred= -18 gold= -18
4 21+78 pred= 99 gold= 99

Solving private-clean...
Solving private-noisy...

Wrote: private-clean.txt and private-noisy.txt
